In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Tuple, Dict
import logging
import tqdm

class CanaryExposureCalculator:
    def __init__(self, model_name: str):
        """
        Args:
            model_name: HuggingFace model identifier
        """
        logging.info(f"Loading model and tokenizer: {model_name}")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        logging.info(f"Model loaded on {self.device}")

    def generate_canaries(self, format_string: str, randomness_space: List, num_canaries: int = 100) -> List[str]:
        """
        Generate canaries based on a format string with placeholders.
        
        Args:
            format_string: String with "_" as placeholders
            randomness_space: List of possible values to fill the placeholders
            num_canaries: Number of canaries to generate
            
        Returns:
            List of generated canary strings
        """
        canaries = []
        
        # Count placeholders in format string
        num_placeholders = format_string.count("_")
        

        for _ in range(num_canaries):
            canary = format_string
            for _ in range(num_placeholders):
                random_value = np.random.choice(randomness_space)
                canary = canary.replace("_", str(random_value), 1)
            canaries.append(canary)
            
        return canaries
    
    def calculate_ranks(self, format_string: str, randomness_space: List, 
                        canaries: List[str]) -> Dict[str, int]:
        """
        Calculate the rank of each canary among all possible completions.
        
        Args:
            format_string: The format string with placeholders
            randomness_space: List of possible values for placeholders
            canaries: List of canary strings to rank
            
        Returns:
            Dictionary mapping each canary to its rank
        """

        prefix = format_string.split("_")[0]
        
        prefix_tokens = self.tokenizer.encode(prefix, return_tensors="pt").to(self.device)
        


        completions = [canary[len(prefix):] for canary in canaries]
        
        # Calculate likelihood of each completion
        canary_scores = {}
        for i, completion in enumerate(tqdm.tqdm(completions)):
            completion_tokens = self.tokenizer.encode(completion, add_special_tokens=False)
            

            score = 0
            current_input = prefix_tokens.clone()
            
            for token in completion_tokens:
                with torch.no_grad():
                    outputs = self.model(current_input)
                    next_token_logits = outputs.logits[:, -1, :]
                    next_token_probs = torch.softmax(next_token_logits, dim=-1)
                    

                    token_prob = next_token_probs[0, token].item()
                    score += np.log(token_prob) if token_prob > 0 else -1000
                    

                    current_input = torch.cat([current_input, 
                                             torch.tensor([[token]]).to(self.device)], dim=1)
            
            canary_scores[canaries[i]] = score
        

        sorted_canaries = sorted(canary_scores.keys(), key=lambda k: canary_scores[k], reverse=True)
        

        canary_ranks = {canary: sorted_canaries.index(canary) + 1 for canary in canaries}
        
        return canary_ranks
    
    def calculate_exposure(self, format_string: str, randomness_space: List, 
                          target_canaries: List[str], num_samples: int = 1000) -> Dict[str, float]:
        """
        Args:
            format_string: The format string with placeholders
            randomness_space: List of possible values for placeholders
            target_canaries: List of canaries to calculate exposure for
            num_samples: Number of random canaries to sample for ranking
            
        Returns:
            Dictionary mapping each target canary to its exposure value
        """
        # Generate additional random canaries
        random_canaries = self.generate_canaries(format_string, randomness_space, num_samples)
        

        all_canaries = target_canaries + random_canaries
        

        canary_ranks = self.calculate_ranks(format_string, randomness_space, all_canaries)
        

        num_placeholders = format_string.count("_")
        space_size = len(randomness_space) ** num_placeholders
        

        exposures = {}
        for canary in target_canaries:
            rank = canary_ranks[canary]
            exposure = np.log2(space_size) - np.log2(rank)
            exposures[canary] = max(0, exposure)  
            
        return exposures


def run():

    calculator = CanaryExposureCalculator("gpt2")
    

    format_string = "The random number is _"
    randomness_space = list(range(9999999))  
    

    target_canaries = ["The random number is 7", "The random number is 3"]
    

    exposures = calculator.calculate_exposure(format_string, randomness_space, target_canaries)
    

    for canary, exposure in exposures.items():
        print(f"Canary: {canary}")
        print(f"Exposure: {exposure:.2f} bits")
        print(f"Interpretation: {'High exposure (likely memorized)' if exposure > 3 else 'Low exposure'}")
        print()

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    run()